---
title: "Machine Learning: Dimensionality Reduction and Sparse Representation"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---



<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/14-dimensionality-sparse-representation.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)



## **Dimensionality Reduction and Sparse Representation**

A representation determines what a learning algorithm can see easily. The same observation might be stored as ten raw measurements, thousands of word counts, millions of image pixels, or a short vector of latent coordinates. **Dimensionality reduction** constructs a lower-dimensional representation that retains selected structure while discarding variation judged redundant, noisy, or irrelevant.

For observations $\mathbf{x}\in\mathbb R^d$, a dimensionality-reduction map produces

$$
\mathbf z=f(\mathbf x),
\qquad
\mathbf z\in\mathbb R^k,
\qquad k\ll d.
$$

The map may be linear, nonlinear, probabilistic, random, or learned by a neural network. A **sparse representation** is different but complementary: its ambient dimension may remain large, yet only a small number of coordinates are active for each observation. A vector with 1,000 possible dictionary atoms and 10 nonzero coefficients is high-dimensional in shape but simple in effective description.

Reduction is never free compression. Every method chooses which relationships to preserve:

- PCA preserves variance and linear least-squares reconstruction;
- SVD preserves the best low-rank matrix approximation;
- factor analysis explains covariance through latent variables and explicit noise;
- ICA seeks statistically independent non-Gaussian sources;
- NMF builds non-negative observations from additive non-negative parts;
- sparse coding reconstructs each observation with a few dictionary atoms;
- manifold methods preserve selected local or geodesic relationships;
- random projection approximately preserves many pairwise distances;
- autoencoders learn nonlinear reconstruction maps under a bottleneck or regularizer.

A useful representation for visualization may be unsuitable for prediction. A representation with low reconstruction error may discard a rare feature that matters for safety. A visually separated embedding may exaggerate gaps that are weak in the original space. This chapter therefore treats evaluation and leakage prevention as part of representation learning, not as an afterthought.



### **Why Reduce Dimensionality?**

High dimension creates computational and statistical difficulties. Storage, matrix operations, neighbour search, and model complexity can all grow with the number of features. More subtly, a fixed sample becomes sparse relative to the volume of its feature space, so local evidence becomes harder to obtain.

#### **The Curse of Dimensionality**

The volume of a $d$-dimensional unit cube is one, but a side length $r<1$ occupies fraction $r^d$. If $r=0.5$, the central subcube occupies $0.5^{10}\approx0.001$ in ten dimensions and $0.5^{50}\approx8.9\times10^{-16}$ in fifty dimensions. To capture the same fraction of observations, a neighbourhood must expand dramatically as $d$ grows.

Distances also become less discriminative under many common data models. If independent features contribute comparable random variation, squared Euclidean distance sums many terms and concentrates around its expectation. The nearest and farthest observations can become relatively similar, undermining nearest-neighbour, kernel, density, and clustering methods. This is not a theorem that all high-dimensional data are hopeless. Sparsity, learned metrics, low intrinsic dimension, and strong structure can make high-dimensional problems tractable.

<details>
<summary><strong>Python: observe distance concentration as dimension grows</strong></summary>

```python
import numpy as np
from scipy.spatial.distance import pdist

rng = np.random.default_rng(14)

for dimension in [2, 10, 50, 200, 1000]:
    X = rng.normal(size=(500, dimension))
    distances = pdist(X, metric="euclidean")
    relative_spread = distances.std() / distances.mean()
    percentile_ratio = np.percentile(distances, 5) / np.percentile(distances, 95)
    print(
        f"d={dimension:4d}",
        "coefficient of variation=", round(relative_spread, 3),
        "p05/p95=", round(percentile_ratio, 3),
    )
```

</details>

The coefficient of variation falls and the low-to-high percentile ratio approaches one. Whether this matters depends on the task, metric, and data-generating structure. Adding 1,000 informative sparse features is not equivalent to adding 1,000 independent noise features.

#### **Compression, Denoising, and Visualization**

Dimensionality reduction serves several distinct goals:

| Goal | Desired preservation | Typical diagnostic |
|---|---|---|
| Compression | Enough information to reconstruct or store efficiently | Held-out reconstruction error, storage ratio |
| Denoising | Stable signal while discarding measurement variation | Residual analysis, performance on clean targets |
| Visualization | Local or global relationships relevant to exploration | Trustworthiness, repeated embeddings, original-feature inspection |
| Faster learning | Task performance with lower time and memory | End-to-end validation against the unreduced baseline |
| Regularization | Lower effective model complexity | Validation performance and stability |
| Interpretation | Components with defensible domain meaning | Loading structure, reproducibility, expert review |

<div class="diagram-scroll">

![Goals and risks of mapping high-dimensional observations into a compact representation.](assets/dimension-reduction-goals.svg){fig-alt="A high-dimensional matrix passes through a representation map to a compact vector used for compression, denoising, visualization, and modelling, with explicit evaluation of information loss."}

</div>

**Intrinsic dimension** is the number of degrees of freedom needed to describe the relevant data structure, which can be much smaller than the number of recorded features. A video frame may have millions of pixels but be governed by a few object positions, lighting conditions, and camera parameters. Estimating intrinsic dimension is difficult, and the relevant dimension can depend on the task: preserving identity, motion, and texture may require different coordinates.

Reduction can also create leakage. If PCA, scaling, manifold learning, or feature selection is fitted on the complete dataset before a train/test split, the test distribution influences the learned representation. Any data-dependent transformation used in predictive evaluation must be fitted inside each training fold and then applied to validation data.



### **Principal Component Analysis**

Principal Component Analysis (PCA) is the fundamental linear dimensionality-reduction method. It finds orthogonal directions along which centred observations vary most, then projects observations onto the first $k$ directions. PCA is descriptive rather than supervised: it does not know which variation predicts a target or carries domain importance.

Let the centred data matrix be $\mathbf X_c\in\mathbb R^{n\times d}$, where each feature mean has been subtracted. Its sample covariance is

$$
\mathbf S=\frac{1}{n-1}\mathbf X_c^\top\mathbf X_c.
$$

#### **Variance Maximization and Reconstruction Error**

The first principal direction $\mathbf w_1$ solves

$$
\mathbf w_1
=\arg\max_{\lVert\mathbf w\rVert_2=1}
\operatorname{Var}(\mathbf X_c\mathbf w)
=\arg\max_{\lVert\mathbf w\rVert_2=1}
\mathbf w^\top\mathbf S\mathbf w.
$$

The unit-norm constraint prevents variance from being increased merely by scaling $\mathbf w$. Using a Lagrange multiplier gives

$$
\mathbf S\mathbf w=\lambda\mathbf w,
$$

so principal directions are covariance eigenvectors, ordered by decreasing eigenvalues. Eigenvalue $\lambda_j$ is the sample variance of component score $\mathbf X_c\mathbf w_j$. Subsequent directions maximize remaining variance subject to orthogonality with previous directions.

For loading matrix $\mathbf W_k=[\mathbf w_1,\ldots,\mathbf w_k]$, scores and reconstruction are

$$
\mathbf Z=\mathbf X_c\mathbf W_k,
\qquad
\widehat{\mathbf X}_c=\mathbf Z\mathbf W_k^\top
=\mathbf X_c\mathbf W_k\mathbf W_k^\top.
$$

The same subspace that maximizes captured variance minimizes squared orthogonal reconstruction error

$$
\left\lVert
\mathbf X_c-\mathbf X_c\mathbf W_k\mathbf W_k^\top
\right\rVert_F^2.
$$

This equivalence explains both PCA's strength and its limitation: low-variance structure is deliberately discarded, even if that structure is predictive or scientifically important.

<div class="diagram-scroll">

![PCA projection onto the maximum-variance direction and its equivalent reconstruction-error view.](assets/pca-variance-reconstruction.svg){fig-alt="Correlated observations are projected orthogonally onto PC1, with residual segments showing discarded variance; a side panel connects variance maximization to reconstruction minimization."}

</div>

<details>
<summary><strong>Python: implement PCA through covariance eigendecomposition</strong></summary>

```python
import numpy as np
from sklearn.decomposition import PCA

rng = np.random.default_rng(22)
latent = rng.normal(size=(500, 2))
mixing = np.array([
    [2.0, 0.2],
    [1.4, -0.7],
    [0.8, 1.5],
    [-0.4, 1.0],
])
X = latent @ mixing.T + rng.normal(scale=0.15, size=(500, 4))

# Centre before constructing the covariance matrix.
mean = X.mean(axis=0)
X_centered = X - mean
covariance = X_centered.T @ X_centered / (len(X) - 1)

# eigh exploits symmetry; reverse the ascending eigenvalue order.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
components = eigenvectors[:, order]

scores = X_centered @ components[:, :2]
reconstruction = scores @ components[:, :2].T + mean
manual_mse = np.mean((X - reconstruction) ** 2)

sklearn_pca = PCA(n_components=2).fit(X)
sklearn_reconstruction = sklearn_pca.inverse_transform(sklearn_pca.transform(X))

print("eigenvalues:", np.round(eigenvalues, 3))
print("explained ratios:", np.round(eigenvalues / eigenvalues.sum(), 3))
print("manual reconstruction MSE:", round(manual_mse, 6))
print("sklearn reconstruction MSE:", round(np.mean((X - sklearn_reconstruction) ** 2), 6))
```

</details>

Principal-vector signs can differ between implementations because $\mathbf w$ and $-\mathbf w$ span the same axis. Compare subspaces, explained variance, and reconstruction rather than expecting identical signs.

#### **PCA through Eigendecomposition and SVD**

PCA can be computed from covariance eigendecomposition, but explicitly forming $\mathbf X_c^\top\mathbf X_c$ can worsen numerical conditioning and cost $O(d^2)$ memory. Applying Singular Value Decomposition directly to $\mathbf X_c$ is often more stable:

$$
\mathbf X_c=\mathbf U\boldsymbol\Sigma\mathbf V^\top.
$$

The columns of $\mathbf V$ are principal directions, and

$$
\lambda_j=\frac{\sigma_j^2}{n-1}.
$$

Modern implementations select among full, covariance-based, truncated, or randomized solvers according to matrix shape and requested component count. Randomized SVD is especially useful when only a small number of leading components is needed.

Feature scaling changes PCA. Centring is essential, but standardization is optional and substantive. PCA on a covariance matrix lets high-variance units dominate; PCA after standardization is equivalent to using a correlation matrix and gives each standardized feature comparable initial variance. Neither is universally correct.

#### **Choosing the Number of Components**

The explained variance ratio of component $j$ is

$$
r_j=\frac{\lambda_j}{\sum_{m=1}^{d}\lambda_m},
$$

and cumulative explained variance is $R_k=\sum_{j=1}^{k}r_j$. A threshold such as 90% or 95% is easy to communicate but arbitrary. Other choices include a scree-plot elbow, held-out reconstruction error, parallel analysis against noise, likelihood criteria for probabilistic PCA, or downstream cross-validation.

<details>
<summary><strong>Python: compare variance, reconstruction, and predictive utility across component counts</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=31
)

for components in [5, 10, 20, 40]:
    pca = PCA(n_components=components, random_state=31).fit(X_train)
    test_scores = pca.transform(X_test)
    reconstructed = pca.inverse_transform(test_scores)
    test_mse = np.mean((X_test - reconstructed) ** 2)

    classifier = make_pipeline(
        StandardScaler(),
        PCA(n_components=components, random_state=31),
        LogisticRegression(max_iter=2000),
    )
    classifier.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, classifier.predict(X_test))
    print(
        f"k={components:2d}",
        "variance=", round(pca.explained_variance_ratio_.sum(), 3),
        "test MSE=", round(test_mse, 3),
        "test accuracy=", round(accuracy, 3),
    )
```

</details>

The component count that reconstructs pixels well need not be the smallest count that preserves classification. If the purpose is prediction, choose dimensionality inside the predictive validation procedure. If the purpose is scientific description, inspect loadings, residuals, stability, and domain plausibility as well as explained variance.



### **Singular Value Decomposition**

Singular Value Decomposition (SVD) is a matrix factorization, not by itself a statistical model. Every real matrix $\mathbf X\in\mathbb R^{n\times d}$ can be written as

$$
\mathbf X=\mathbf U\boldsymbol\Sigma\mathbf V^\top,
$$

where the columns of $\mathbf U$ are orthonormal left singular vectors, the columns of $\mathbf V$ are orthonormal right singular vectors, and $\boldsymbol\Sigma$ contains non-negative singular values

$$
\sigma_1\geq\sigma_2\geq\cdots\geq0.
$$

For rank $r$, the compact SVD has $\mathbf U\in\mathbb R^{n\times r}$, $\boldsymbol\Sigma\in\mathbb R^{r\times r}$, and $\mathbf V\in\mathbb R^{d\times r}$. The triplet $(\mathbf u_j,\sigma_j,\mathbf v_j)$ contributes the rank-one matrix $\sigma_j\mathbf u_j\mathbf v_j^\top$.

<div class="diagram-scroll">

![The observation, strength, and feature-direction factors in SVD.](assets/svd-three-factor-view.svg){fig-alt="A data matrix X is decomposed into U sample coordinates, Sigma singular values, and V-transpose feature directions; retaining the first k triplets produces a rank-k approximation."}

</div>

#### **Low-Rank Approximation**

Truncating after $k$ singular values gives

$$
\mathbf X_k
=\sum_{j=1}^{k}\sigma_j\mathbf u_j\mathbf v_j^\top
=\mathbf U_k\boldsymbol\Sigma_k\mathbf V_k^\top.
$$

The Eckart-Young-Mirsky theorem states that $\mathbf X_k$ is the best rank-$k$ approximation under the Frobenius norm and spectral norm. Under squared Frobenius loss,

$$
\left\lVert\mathbf X-\mathbf X_k\right\rVert_F^2
=\sum_{j=k+1}^{r}\sigma_j^2.
$$

This exact identity separates approximation error into discarded singular directions. It underlies image compression, latent semantic analysis, matrix completion methods, and efficient linear algebra.

<details>
<summary><strong>Python: verify optimal rank-k reconstruction error</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(40)
# Create a nearly rank-4 matrix, then add small full-rank noise.
left = rng.normal(size=(120, 4))
right = rng.normal(size=(4, 70))
X = left @ right + rng.normal(scale=0.08, size=(120, 70))

U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)

for rank in [1, 2, 4, 8, 20]:
    reconstruction = (U[:, :rank] * singular_values[:rank]) @ Vt[:rank]
    observed_squared_error = np.linalg.norm(X - reconstruction, ord="fro") ** 2
    theoretical_squared_error = np.sum(singular_values[rank:] ** 2)
    print(
        f"rank={rank:2d}",
        "observed=", round(observed_squared_error, 6),
        "discarded singular energy=", round(theoretical_squared_error, 6),
    )
```

</details>

PCA is obtained by applying SVD to a **centred** data matrix. TruncatedSVD in scikit-learn intentionally does not centre its input, so it can operate efficiently on sparse matrices whose centring would destroy sparsity. In text mining, applying TruncatedSVD to a term-document matrix is commonly called **Latent Semantic Analysis (LSA)**. The resulting directions mix overall term frequency and variation because the matrix is not centred.

<details>
<summary><strong>Python: apply TruncatedSVD to a sparse text matrix</strong></summary>

```python
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "graph algorithms find shortest paths and spanning trees",
    "network flow algorithms optimize capacity through graphs",
    "sorting and searching are fundamental algorithmic problems",
    "neural networks learn layered representations from data",
    "gradient descent trains neural models with backpropagation",
    "representation learning supports language and vision models",
    "probability models quantify uncertainty and latent variables",
    "bayesian inference updates probability distributions with evidence",
]

vectorizer = TfidfVectorizer(stop_words="english")
X_sparse = vectorizer.fit_transform(documents)
svd = TruncatedSVD(n_components=3, random_state=40).fit(X_sparse)
coordinates = svd.transform(X_sparse)
terms = vectorizer.get_feature_names_out()

for component_index, component in enumerate(svd.components_):
    top_terms = terms[component.argsort()[-5:][::-1]]
    print(f"component {component_index + 1}:", top_terms.tolist())

print("sparse input shape:", X_sparse.shape)
print("reduced shape:", coordinates.shape)
print("explained variance ratio sum:", round(svd.explained_variance_ratio_.sum(), 3))
```

</details>

SVD components are algebraically optimal for low-rank squared error, but they are not automatically semantic topics. Components can contain positive and negative loadings, signs can flip, and correlated latent concepts can rotate into mixtures.



### **Factor Analysis and Independent Component Analysis**

PCA, Factor Analysis (FA), and Independent Component Analysis (ICA) all produce linear latent coordinates, but they encode different assumptions. PCA finds a descriptive subspace of maximum variance. FA is a probabilistic covariance model with explicit feature-specific noise. ICA seeks latent sources that are statistically independent rather than merely uncorrelated.

#### **Latent Factors**

Factor analysis assumes

$$
\mathbf x=\boldsymbol\mu+\boldsymbol\Lambda\mathbf z+\boldsymbol\epsilon,
$$

where $\mathbf z\sim\mathcal N(\mathbf0,\mathbf I_k)$ contains latent factors, $\boldsymbol\Lambda\in\mathbb R^{d\times k}$ is a loading matrix, and

$$
\boldsymbol\epsilon\sim\mathcal N(\mathbf0,\boldsymbol\Psi)
$$

is feature-specific noise with diagonal covariance $\boldsymbol\Psi$. Therefore,

$$
\operatorname{Cov}(\mathbf x)
=\boldsymbol\Lambda\boldsymbol\Lambda^\top+\boldsymbol\Psi.
$$

The low-rank term explains shared covariance, while $\boldsymbol\Psi$ allows each observed feature to contain unique variation. PCA does not make this shared-versus-unique decomposition; its reconstruction residual is orthogonal to the selected subspace rather than a fitted diagonal noise model.

Factor loadings are not uniquely oriented. If $\mathbf R$ is orthogonal, then $\boldsymbol\Lambda\mathbf R$ gives the same covariance because

$$
(\boldsymbol\Lambda\mathbf R)(\boldsymbol\Lambda\mathbf R)^\top
=\boldsymbol\Lambda\boldsymbol\Lambda^\top.
$$

Rotations such as varimax seek a simpler loading pattern, but they add an interpretive criterion rather than discovering a uniquely true orientation. Factor number should be evaluated with held-out likelihood, residual correlations, parallel analysis, stability, and domain interpretability.

<div class="diagram-scroll">

![How factor analysis explains shared covariance and ICA unmixes independent sources.](assets/factor-analysis-ica.svg){fig-alt="Factor analysis maps latent variables into correlated observed features plus feature-specific noise, while ICA learns an unmixing matrix to recover independent non-Gaussian source signals."}

</div>

<details>
<summary><strong>Python: recover a latent factor subspace and unique variances</strong></summary>

```python
import numpy as np
from scipy.linalg import subspace_angles
from sklearn.decomposition import FactorAnalysis, PCA
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(51)
true_loadings = np.array([
    [1.4, 0.0], [1.1, 0.1], [0.8, -0.1],
    [0.0, 1.3], [0.1, 1.0], [-0.1, 0.7],
])
unique_std = np.array([0.25, 0.35, 0.50, 0.20, 0.45, 0.65])
latent = rng.normal(size=(1600, 2))
X = latent @ true_loadings.T + rng.normal(size=(1600, 6)) * unique_std
X_train, X_test = train_test_split(X, test_size=0.30, random_state=51)

fa = FactorAnalysis(n_components=2, rotation="varimax", random_state=51).fit(X_train)
pca = PCA(n_components=2, random_state=51).fit(X_train)

angles = np.degrees(subspace_angles(true_loadings, fa.components_.T))
print("FA principal angles to true loading space:", np.round(angles, 2))
print("true unique variances:", np.round(unique_std**2, 3))
print("estimated unique variances:", np.round(fa.noise_variance_, 3))
print("FA mean held-out log-likelihood:", round(fa.score(X_test), 3))
print("PCA explained variance ratio:", round(pca.explained_variance_ratio_.sum(), 3))
```

</details>

Subspace angles evaluate the span rather than individual rotated columns. In real data, true loadings are unavailable, so residual covariance and out-of-sample likelihood become more important.

#### **Statistical Independence**

ICA assumes observed signals are linear mixtures of latent sources:

$$
\mathbf x=\mathbf A\mathbf s,
$$

and estimates an unmixing matrix $\mathbf W\approx\mathbf A^{-1}$ so that $\widehat{\mathbf s}=\mathbf W\mathbf x$ has mutually independent components. Independence is stronger than zero correlation. For Gaussian variables, decorrelation exhausts the available second-order information, so ICA requires at most one Gaussian source and exploits non-Gaussianity through measures such as negentropy or kurtosis.

ICA is useful for blind source separation, artifact removal in EEG, and mixtures of sensor signals. Component order and scale are unidentifiable: multiplying one source by $c$ and dividing the corresponding mixing column by $c$ leaves observations unchanged. Evaluation therefore aligns estimated sources by permutation and sign before comparison.

<details>
<summary><strong>Python: separate mixed non-Gaussian signals with FastICA</strong></summary>

```python
import numpy as np
from scipy import signal
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import FastICA, PCA

rng = np.random.default_rng(63)
time = np.linspace(0, 8, 2000)
sources = np.column_stack([
    np.sin(2 * time),
    np.sign(np.sin(3 * time)),
    signal.sawtooth(2 * np.pi * time),
])
sources += 0.05 * rng.normal(size=sources.shape)
sources /= sources.std(axis=0)

mixing = np.array([[1.0, 1.0, 0.5], [0.5, 2.0, 1.0], [1.5, 1.0, 2.0]])
observations = sources @ mixing.T

ica_sources = FastICA(
    n_components=3,
    whiten="unit-variance",
    random_state=63,
    max_iter=1000,
).fit_transform(observations)
pca_scores = PCA(n_components=3).fit_transform(observations)

def matched_absolute_correlation(reference, estimated):
    correlations = np.corrcoef(reference.T, estimated.T)[:3, 3:]
    rows, cols = linear_sum_assignment(-np.abs(correlations))
    return np.abs(correlations[rows, cols])

print("ICA matched correlations:", np.round(matched_absolute_correlation(sources, ica_sources), 3))
print("PCA matched correlations:", np.round(matched_absolute_correlation(sources, pca_scores), 3))
```

</details>

PCA decorrelates the mixtures but generally does not recover independent non-Gaussian sources. ICA can recover them when its linear, independent-source, and adequate-sampling assumptions are plausible.



### **Non-Negative Matrix Factorization**

Non-Negative Matrix Factorization (NMF) applies when the data matrix is non-negative, as with pixel intensity, word counts, power spectra, or transaction amounts. It approximates

$$
\mathbf X\approx\mathbf W\mathbf H,
\qquad
\mathbf W\geq0,
\quad
\mathbf H\geq0,
$$

where $\mathbf X\in\mathbb R_+^{n\times d}$, $\mathbf W\in\mathbb R_+^{n\times k}$ contains observation coefficients, and $\mathbf H\in\mathbb R_+^{k\times d}$ contains component patterns.

#### **Parts-Based Representations**

Because negative cancellation is forbidden, an observation is reconstructed by **adding** active components. For face images, components may resemble eyes, mouths, or lighting patterns; for text, they may resemble groups of terms. This parts-based interpretation is a tendency induced by constraints, not a guarantee of semantic truth.

<div class="diagram-scroll">

![An observation reconstructed as an additive combination of non-negative NMF parts.](assets/nmf-additive-parts.svg){fig-alt="A non-negative face-like observation is approximated by positive weights multiplying eye, mouth, and other non-negative component atoms."}

</div>

A common objective is

$$
\min_{\mathbf W,\mathbf H\geq0}
\frac{1}{2}\lVert\mathbf X-\mathbf W\mathbf H\rVert_F^2
+\lambda_W\Omega(\mathbf W)
+\lambda_H\Omega(\mathbf H).
$$

Regularizers can encourage sparse coefficients or components. Other losses include generalized Kullback-Leibler divergence, which is related to a Poisson observation model and can suit count-like data. The joint problem is non-convex, although fixing either factor makes the other subproblem easier. Initialization, component count, solver, and regularization can therefore change the result.

NMF has scale and permutation ambiguities. Multiplying column $k$ of $\mathbf W$ by $c>0$ and dividing row $k$ of $\mathbf H$ by $c$ leaves the product unchanged. Component order is also arbitrary. Normalize components before comparing magnitudes across runs.

<details>
<summary><strong>Python: extract additive term components from non-negative text features</strong></summary>

```python
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "graph search shortest path spanning tree network flow",
    "graph algorithms optimize paths trees and network capacity",
    "dynamic programming and greedy algorithms solve optimization problems",
    "neural networks learn representations with gradient descent",
    "deep neural models use backpropagation and nonlinear layers",
    "representation learning supports language models and computer vision",
    "bayesian probability models represent uncertainty and prior knowledge",
    "probabilistic inference estimates latent variables and distributions",
    "likelihood and posterior distributions support statistical inference",
]

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)
nmf = NMF(
    n_components=3,
    init="nndsvda",
    random_state=72,
    max_iter=1000,
).fit(X)
document_weights = nmf.transform(X)
terms = vectorizer.get_feature_names_out()

for component_index, component in enumerate(nmf.components_):
    top_terms = terms[component.argsort()[-6:][::-1]]
    print(f"component {component_index + 1}:", top_terms.tolist())

print("all component entries non-negative:", bool((nmf.components_ >= 0).all()))
print("document representation shape:", document_weights.shape)
print("reconstruction error:", round(nmf.reconstruction_err_, 3))
```

</details>

Top terms help name a component, but naming should follow stability checks across seeds, component counts, samples, and preprocessing. TF-IDF is non-negative, yet its weighting affects which additive patterns NMF can discover.



### **Dictionary Learning and Sparse Coding**

Dictionary learning represents each observation as a linear combination of learned **atoms**. Unlike PCA, the dictionary can be overcomplete: it may contain more atoms than observed dimensions. Simplicity comes from using only a few atoms for each observation.

Let $\mathbf D\in\mathbb R^{m\times d}$ contain $m$ row atoms and let $\boldsymbol\alpha_i\in\mathbb R^m$ encode observation $\mathbf x_i$. Sparse coding solves a problem such as

$$
\min_{\boldsymbol\alpha_i}
\frac{1}{2}\lVert\mathbf x_i-\boldsymbol\alpha_i\mathbf D\rVert_2^2
+\lambda\lVert\boldsymbol\alpha_i\rVert_1.
$$

The reconstruction term rewards fidelity. The $L_1$ penalty encourages many coefficients to become exactly or approximately zero. A larger $\lambda$ produces simpler but less accurate reconstructions. When the desired nonzero count is known, Orthogonal Matching Pursuit (OMP) greedily selects atoms instead of using an $L_1$ penalty.

#### **Sparse Representations**

<div class="diagram-scroll">

![A signal reconstructed from a few active atoms in an overcomplete dictionary.](assets/sparse-dictionary-coding.svg){fig-alt="A signal is mapped through a dictionary containing several waveform atoms to a coefficient vector in which only a few bars are nonzero."}

</div>

Sparsity can make representation efficient and locally interpretable: different observations activate different small subsets of reusable patterns. Applications include image patches, audio events, compressed sensing, and feature construction. Sparse does not automatically mean causal or semantically pure, and correlated atoms can make individual codes unstable.

Dictionary learning jointly optimizes codes $\mathbf A$ and atoms $\mathbf D$:

$$
\min_{\mathbf A,\mathbf D}
\frac{1}{2}\lVert\mathbf X-\mathbf A\mathbf D\rVert_F^2
+\lambda\lVert\mathbf A\rVert_{1,1},
\qquad
\lVert\mathbf d_j\rVert_2\leq1.
$$

The atom norm constraint prevents an arbitrary scaling in which dictionary atoms grow while coefficients shrink to reduce the penalty. Algorithms commonly alternate between coding observations with fixed atoms and updating atoms with fixed codes. The joint problem remains non-convex.

#### **L1-Regularized Reconstruction**

The $L_1$ ball has corners aligned with coordinate axes, so a least-squares contour often touches it at a point with zero coordinates. By contrast, an $L_2$ penalty shrinks all coefficients smoothly and rarely produces exact zeros. This geometric distinction is the same one underlying Lasso regression.

<details>
<summary><strong>Python: compare dense and sparse reconstruction codes</strong></summary>

```python
import numpy as np
from sklearn.decomposition import sparse_encode

rng = np.random.default_rng(84)
n_samples, n_features, n_atoms = 300, 25, 45

# Create an overcomplete dictionary with unit-norm atoms.
dictionary = rng.normal(size=(n_atoms, n_features))
dictionary /= np.linalg.norm(dictionary, axis=1, keepdims=True)

# Generate observations from exactly three active atoms each.
true_codes = np.zeros((n_samples, n_atoms))
for row in range(n_samples):
    active = rng.choice(n_atoms, size=3, replace=False)
    true_codes[row, active] = rng.normal(size=3)
X = true_codes @ dictionary + rng.normal(scale=0.02, size=(n_samples, n_features))

# Minimum-norm least squares uses a dense code in an overcomplete system.
dense_codes = X @ np.linalg.pinv(dictionary)
omp_codes = sparse_encode(
    X,
    dictionary,
    algorithm="omp",
    n_nonzero_coefs=3,
)
lasso_codes = sparse_encode(
    X,
    dictionary,
    algorithm="lasso_cd",
    alpha=0.04,
    max_iter=2000,
)

for name, codes in [("dense", dense_codes), ("OMP", omp_codes), ("L1", lasso_codes)]:
    reconstruction = codes @ dictionary
    nonzeros = np.mean(np.sum(np.abs(codes) > 1e-6, axis=1))
    mse = np.mean((X - reconstruction) ** 2)
    print(name, "mean nonzeros=", round(nonzeros, 2), "MSE=", round(mse, 6))
```

</details>

OMP enforces the requested support size, whereas L1 chooses support through a penalty. Recovery of the true atoms or support requires conditions on incoherence, sparsity, noise, and sample size; low reconstruction error alone does not establish correct latent structure.



### **Manifold Learning**

Manifold learning assumes that high-dimensional observations lie near a lower-dimensional curved surface. A three-dimensional Swiss roll, for example, has two intrinsic coordinates even though straight-line distances through ambient space can connect points that are far apart along the sheet.

These methods usually begin with a local neighbour graph. Their success therefore depends on feature representation, metric, sample coverage, and neighbourhood size. Too few neighbours disconnect the graph; too many create shortcuts across folds. Unlike PCA, most manifold embeddings are nonlinear and do not provide simple feature loadings or a globally valid inverse transformation.

#### **Isomap and Locally Linear Embedding**

**Isomap** attempts to preserve geodesic distance along a manifold:

1. connect each observation to its $k$ nearest neighbours or to points within a radius;
2. weight edges by input-space distance;
3. approximate geodesic distances with all-pairs shortest paths on the graph;
4. apply classical multidimensional scaling to the geodesic-distance matrix.

If $d_G(i,j)$ is graph shortest-path distance, Isomap finds low-dimensional coordinates whose Euclidean distances approximate $d_G(i,j)$. It can recover global manifold geometry when sampling is dense and the neighbour graph contains neither disconnections nor false shortcuts. Noise, holes, multiple manifolds, and high computational cost can undermine it.

**Locally Linear Embedding (LLE)** preserves local reconstruction relationships rather than global path lengths. For each observation, it first finds weights that reconstruct it from neighbours:

$$
\min_{\{w_{ij}\}}
\sum_i
\left\lVert
\mathbf x_i-\sum_{j\in N(i)}w_{ij}\mathbf x_j
\right\rVert_2^2,
\qquad
\sum_{j\in N(i)}w_{ij}=1.
$$

It then finds low-dimensional coordinates $\mathbf z_i$ preserving the same weights:

$$
\min_{\{\mathbf z_i\}}
\sum_i
\left\lVert
\mathbf z_i-\sum_{j\in N(i)}w_{ij}\mathbf z_j
\right\rVert_2^2,
$$

subject to centring and scale constraints that avoid a trivial all-zero solution. LLE can unfold locally linear surfaces but is sensitive to neighbour count, regularization, and uneven sampling.

<div class="diagram-scroll">

![How Isomap and LLE preserve different relationships while unfolding a manifold.](assets/manifold-methods-map.svg){fig-alt="A curved manifold becomes a neighbour graph; Isomap preserves shortest graph paths, LLE preserves local reconstruction weights, and both seek a lower-dimensional unfolding."}

</div>

<details>
<summary><strong>Python: compare PCA, Isomap, and LLE on an S-curve</strong></summary>

```python
from sklearn.datasets import make_s_curve
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap, LocallyLinearEmbedding, trustworthiness
from sklearn.preprocessing import StandardScaler

X, intrinsic_position = make_s_curve(n_samples=900, noise=0.03, random_state=95)
X = StandardScaler().fit_transform(X)

embeddings = {
    "PCA": PCA(n_components=2).fit_transform(X),
    "Isomap": Isomap(n_neighbors=12, n_components=2).fit_transform(X),
    "LLE": LocallyLinearEmbedding(
        n_neighbors=12,
        n_components=2,
        method="standard",
        random_state=95,
    ).fit_transform(X),
}

for name, embedding in embeddings.items():
    local_trust = trustworthiness(X, embedding, n_neighbors=12)
    # Correlation with the known generator coordinate is only available here
    # because this is a controlled synthetic example.
    coordinate_correlation = max(
        abs(__import__("numpy").corrcoef(intrinsic_position, embedding[:, axis])[0, 1])
        for axis in range(2)
    )
    print(
        name,
        "trustworthiness=", round(local_trust, 3),
        "best latent-coordinate correlation=", round(coordinate_correlation, 3),
    )
```

</details>

Trustworthiness checks whether low-dimensional neighbours were genuine neighbours in the input space. It does not establish preservation of global distances, topology, density, or semantic meaning.

#### **t-SNE and UMAP**

**t-distributed Stochastic Neighbor Embedding (t-SNE)** converts high-dimensional neighbourhood affinities into probabilities $p_{ij}$ and low-dimensional affinities into heavy-tailed probabilities $q_{ij}$. It minimizes

$$
\operatorname{KL}(P\Vert Q)
=\sum_{i\ne j}p_{ij}\log\frac{p_{ij}}{q_{ij}}.
$$

Because this KL direction heavily penalizes a true high-dimensional neighbour placed far away, t-SNE emphasizes local neighbourhoods. A Student-$t$ kernel in the embedding reduces the crowding problem by permitting more distant pairs. **Perplexity** controls an effective neighbourhood scale; early exaggeration, learning rate, initialization, and random seed also affect the map.

**UMAP** constructs a weighted fuzzy neighbour graph in the input space and optimizes a low-dimensional graph with a cross-entropy-like objective. `n_neighbors` controls the balance between local detail and broader structure; `min_dist` controls how tightly points may pack in the embedding; `metric` defines input-space similarity. UMAP often scales well and supports transformation of new observations, but it remains model- and parameter-dependent.

<div class="diagram-scroll">

![A guide to trustworthy and misleading interpretations of t-SNE and UMAP.](assets/tsne-umap-interpretation.svg){fig-alt="Local neighbourhood relationships are contrasted with potentially misleading cluster size, spacing, and density, while a second panel shows sensitivity to t-SNE and UMAP settings."}

</div>

Do not interpret cluster area as population variance, inter-cluster gaps as calibrated separation, or embedding density as original density. Colouring by known labels is useful for diagnosis, but labels should not be treated as discovered structure. Fit preprocessing and any transformable embedding on training data only when the representation feeds a predictive model.

<details>
<summary><strong>Python: evaluate t-SNE neighbourhood preservation and seed sensitivity</strong></summary>

```python
import numpy as np
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler

X, labels = load_digits(return_X_y=True)
rng = np.random.default_rng(105)
subset = rng.choice(len(X), size=800, replace=False)
X = StandardScaler().fit_transform(X[subset])

# PCA pre-reduction removes noise and makes t-SNE cheaper.
X_30 = PCA(n_components=30, random_state=105).fit_transform(X)

embeddings = []
for seed in [105, 106]:
    embedding = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate="auto",
        init="pca",
        max_iter=750,
        random_state=seed,
    ).fit_transform(X_30)
    embeddings.append(embedding)
    print("seed", seed, "trustworthiness:", round(trustworthiness(X_30, embedding, n_neighbors=15), 3))

# Pairwise-distance correlation reveals that global geometry can change.
pair_sample = rng.choice(len(X), size=250, replace=False)
distance_correlation = spearmanr(
    pdist(embeddings[0][pair_sample]),
    pdist(embeddings[1][pair_sample]),
).statistic
print("cross-seed global distance correlation:", round(distance_correlation, 3))
```

</details>

UMAP is provided by the optional `umap-learn` package rather than scikit-learn. The following example remains safe in environments where it is not installed.

<details>
<summary><strong>Python: run UMAP when the optional dependency is available</strong></summary>

```python
import importlib.util

if importlib.util.find_spec("umap") is None:
    print("Optional package missing: install 'umap-learn' to run this example.")
else:
    import umap
    from sklearn.datasets import load_digits
    from sklearn.manifold import trustworthiness
    from sklearn.preprocessing import StandardScaler

    X, labels = load_digits(return_X_y=True)
    X = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(
        n_neighbors=20,
        min_dist=0.15,
        metric="euclidean",
        random_state=105,
    )
    embedding = reducer.fit_transform(X)
    print("embedding shape:", embedding.shape)
    print("trustworthiness:", round(trustworthiness(X, embedding, n_neighbors=15), 3))
```

</details>

For both t-SNE and UMAP, rerun plausible hyperparameters and seeds, quantify neighbour preservation, inspect unstable observations, and return to original features before assigning semantic names to visual groups.



### **Random Projection**

Random projection reduces dimension by multiplying observations by a randomly generated matrix rather than learning directions from the data. For $\mathbf X\in\mathbb R^{n\times d}$ and projection $\mathbf R\in\mathbb R^{k\times d}$,

$$
\mathbf Z=\mathbf X\mathbf R^\top.
$$

Entries of $\mathbf R$ may be Gaussian or use a sparse signed distribution. Appropriate scaling keeps expected lengths stable. Projection is fast, streaming-friendly, and avoids iterative optimization; its axes are usually not directly interpretable.

The Johnson-Lindenstrauss lemma states that a finite set of $n$ points can be embedded into a dimension depending logarithmically on $n$ while approximately preserving all pairwise squared distances. With high probability,

$$
(1-\varepsilon)\lVert\mathbf u-\mathbf v\rVert_2^2
\leq
\lVert\mathbf R\mathbf u-\mathbf R\mathbf v\rVert_2^2
\leq
(1+\varepsilon)\lVert\mathbf u-\mathbf v\rVert_2^2.
$$

A common sufficient bound is

$$
k\geq
\frac{4\log n}{\varepsilon^2/2-\varepsilon^3/3}.
$$

The bound is conservative and independent of the original dimension $d$. Smaller distortion $\varepsilon$ requires more projected dimensions.

<div class="diagram-scroll">

![Random projection and approximate pairwise-distance preservation.](assets/random-projection-jl.svg){fig-alt="High-dimensional observations are multiplied by a random matrix to produce k coordinates, with a Johnson-Lindenstrauss inequality showing approximate squared-distance preservation."}

</div>

<details>
<summary><strong>Python: measure pairwise-distance distortion after sparse random projection</strong></summary>

```python
import numpy as np
from sklearn.random_projection import SparseRandomProjection, johnson_lindenstrauss_min_dim

rng = np.random.default_rng(116)
n_samples, n_features = 500, 2000
X = rng.normal(size=(n_samples, n_features))

epsilon = 0.50
safe_dimension = int(johnson_lindenstrauss_min_dim(n_samples, eps=epsilon))
projector = SparseRandomProjection(
    n_components=safe_dimension,
    random_state=116,
)
Z = projector.fit_transform(X)

# Sample pairs instead of materializing every pairwise distance.
left = rng.integers(0, n_samples, size=6000)
right = rng.integers(0, n_samples, size=6000)
valid = left != right
left, right = left[valid], right[valid]
original_squared = np.sum((X[left] - X[right]) ** 2, axis=1)
projected_squared = np.sum((Z[left] - Z[right]) ** 2, axis=1)
relative_distortion = projected_squared / original_squared - 1

print("original dimension:", n_features)
print("JL sufficient dimension:", safe_dimension)
print("median absolute distortion:", round(np.median(np.abs(relative_distortion)), 3))
print("95th percentile absolute distortion:", round(np.percentile(np.abs(relative_distortion), 95), 3))
print("fraction within epsilon:", round(np.mean(np.abs(relative_distortion) <= epsilon), 3))
```

</details>

Random projection is attractive when distances matter, data are extremely wide, and learning an interpretable basis is unnecessary. PCA often achieves lower reconstruction error at the same dimension because it adapts to the sample, but random projection can be much cheaper and can preserve geometry without fitting on all observations. The random matrix and seed must be retained to transform future data consistently.



### **Autoencoders as a Neural Extension**

An autoencoder learns an encoder $f_\theta$ and decoder $g_\phi$ by minimizing reconstruction loss:

$$
\mathbf z=f_\theta(\mathbf x),
\qquad
\widehat{\mathbf x}=g_\phi(\mathbf z),
$$

$$
\min_{\theta,\phi}
\sum_i
\mathcal L\left(
\mathbf x_i,
g_\phi(f_\theta(\mathbf x_i))
\right).
$$

For real-valued standardized inputs, $\mathcal L$ may be mean squared error. Binary data may use Bernoulli cross-entropy, and count or image models may require a likelihood matched to their observation process.

An **undercomplete autoencoder** has a bottleneck dimension smaller than the input. A linear undercomplete autoencoder with squared loss and suitable optimization learns the same principal subspace as PCA, although coordinates can be rotated. Nonlinear activations allow curved reconstruction maps, but also make optimization, identifiability, and generalization more difficult.

<div class="diagram-scroll">

![Encoder, bottleneck, decoder, and regularization options in an autoencoder.](assets/autoencoder-bottleneck.svg){fig-alt="An input passes through decreasing encoder widths into a small latent bottleneck and then through a decoder to a reconstruction, with denoising, sparsity, and regularization constraints listed."}

</div>

An overcomplete network can simply learn an identity map unless constrained. Common variants include:

- **denoising autoencoders**, which reconstruct clean inputs from corrupted versions;
- **sparse autoencoders**, which penalize widespread latent activation;
- **contractive autoencoders**, which penalize encoder sensitivity to small input changes;
- **variational autoencoders**, which learn a regularized probabilistic latent distribution and a generative decoder.

Good reconstruction does not guarantee useful semantics. A model may spend capacity on background texture, memorize training examples, or ignore a low-variance target signal. Evaluate on held-out observations and against a simple PCA baseline.

<details>
<summary><strong>Python: compare a nonlinear bottleneck autoencoder with PCA</strong></summary>

```python
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(128)
latent = rng.uniform(-2.5, 2.5, size=(1200, 2))
# A nonlinear 10-dimensional observation generated from two variables.
X = np.column_stack([
    latent[:, 0], latent[:, 1],
    latent[:, 0] ** 2, latent[:, 1] ** 2,
    latent[:, 0] * latent[:, 1],
    np.sin(latent[:, 0]), np.cos(latent[:, 1]),
    np.sin(latent[:, 0] + latent[:, 1]),
    np.exp(-0.5 * latent[:, 0] ** 2),
    np.exp(-0.5 * latent[:, 1] ** 2),
])
X += rng.normal(scale=0.03, size=X.shape)
X_train, X_test = train_test_split(X, test_size=0.30, random_state=128)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=2).fit(X_train_scaled)
pca_reconstruction = pca.inverse_transform(pca.transform(X_test_scaled))

autoencoder = MLPRegressor(
    hidden_layer_sizes=(14, 2, 14),
    activation="tanh",
    solver="adam",
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=40,
    learning_rate_init=0.005,
    alpha=1e-4,
    max_iter=2000,
    random_state=128,
).fit(X_train_scaled, X_train_scaled)
autoencoder_reconstruction = autoencoder.predict(X_test_scaled)

# Apply the first two fitted hidden layers to obtain bottleneck coordinates.
hidden = np.tanh(X_test_scaled @ autoencoder.coefs_[0] + autoencoder.intercepts_[0])
bottleneck = np.tanh(hidden @ autoencoder.coefs_[1] + autoencoder.intercepts_[1])

print("PCA test reconstruction MSE:", round(mean_squared_error(X_test_scaled, pca_reconstruction), 4))
print("autoencoder test reconstruction MSE:", round(mean_squared_error(X_test_scaled, autoencoder_reconstruction), 4))
print("bottleneck shape:", bottleneck.shape)
```

</details>

This example uses scikit-learn to expose the bottleneck mechanics without introducing a deep-learning dependency. Production autoencoders usually use PyTorch, TensorFlow, or JAX together with mini-batches, validation checkpoints, and architecture-specific regularization.



### **Evaluating and Choosing Representations**

A reduced representation should be evaluated against its intended use. There is no universal score because reconstruction, neighbour preservation, prediction, interpretability, and compression are different objectives.

<div class="diagram-scroll">

![A multi-lens framework for evaluating a learned representation.](assets/representation-evaluation-map.svg){fig-alt="A candidate training-only representation is evaluated using held-out reconstruction, geometric preservation, perturbation stability, downstream utility, and domain meaning or risk."}

</div>

Useful evaluation lenses include:

| Lens | Questions | Example diagnostics |
|---|---|---|
| Reconstruction | Can held-out observations be recovered, and where are residuals concentrated? | MSE, likelihood, residual plots by feature and subgroup |
| Geometry | Which local or global relationships survive? | Trustworthiness, neighbour overlap, distance correlation, stress |
| Stability | Does the representation persist across samples, seeds, and plausible preprocessing? | Principal angles, aligned component correlations, neighbour consistency |
| Downstream utility | Does reduction improve the actual validated task? | Nested-CV performance, latency, memory, calibration |
| Interpretation | Can components be explained without post-hoc storytelling? | Loading sparsity, domain review, reproducibility |
| Risk | Are rare groups or safety-relevant directions erased or distorted? | Subgroup residuals, fairness checks, failure-case inspection |

Reconstruction error should be computed on held-out observations. For PCA-like methods, compare subspaces rather than component signs. For t-SNE or UMAP, examine local neighbourhood preservation and rerun settings rather than comparing axes. For sparse representations, report support stability and reconstruction. For downstream prediction, place the reducer inside the validation pipeline.

<details>
<summary><strong>Python: select PCA dimensionality without leaking test information</strong></summary>

```python
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=140
)

# Scaling and PCA are refitted inside every training fold.
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("reduce", PCA(random_state=140)),
    ("classify", LogisticRegression(max_iter=2000)),
])
search = GridSearchCV(
    pipeline,
    param_grid={
        "reduce__n_components": [10, 20, 30, 40],
        "classify__C": [0.1, 1.0],
    },
    cv=4,
    scoring="accuracy",
    n_jobs=1,
)
search.fit(X_train, y_train)
test_accuracy = accuracy_score(y_test, search.predict(X_test))

print("selected parameters:", search.best_params_)
print("cross-validated accuracy:", round(search.best_score_, 3))
print("untouched test accuracy:", round(test_accuracy, 3))
```

</details>

The test set is used once after the component count and classifier regularization have been selected. If PCA had been fitted before the split, test-set covariance would have influenced the representation and made the estimate optimistic.

<details>
<summary><strong>Python: quantify PCA subspace stability across bootstrap samples</strong></summary>

```python
import numpy as np
from scipy.linalg import subspace_angles
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X, _ = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
reference = PCA(n_components=10, random_state=151).fit(X).components_.T

rng = np.random.default_rng(151)
largest_angles = []
median_angles = []
for _ in range(30):
    indexes = rng.integers(0, len(X), size=len(X))
    bootstrap_subspace = PCA(n_components=10).fit(X[indexes]).components_.T
    angles = np.degrees(subspace_angles(reference, bootstrap_subspace))
    largest_angles.append(angles.max())
    median_angles.append(np.median(angles))

print("median angle across directions:", round(float(np.median(median_angles)), 2), "degrees")
print("median largest principal angle:", round(float(np.median(largest_angles)), 2), "degrees")
print("95th percentile:", round(float(np.percentile(largest_angles, 95)), 2), "degrees")
```

</details>

Most leading directions may be stable while the weakest direction at the selected dimensionality boundary is unstable, especially when neighbouring eigenvalues are nearly tied. Reporting both typical and largest principal angles exposes that distinction without requiring arbitrary sign or component matching.

| Need | Strong starting point | Main caution |
|---|---|---|
| Linear compression and inspectable loadings | PCA | High variance is not necessarily task relevance |
| Sparse term-document reduction | TruncatedSVD | Uncentred components mix mean and variation |
| Shared covariance plus feature-specific noise | Factor Analysis | Rotation and factor count are not uniquely determined |
| Independent non-Gaussian sources | ICA | Strong assumptions; order and scale are unidentified |
| Additive non-negative parts | NMF | Local optima and component instability |
| Few reusable atoms per observation | Dictionary learning / sparse coding | Correlated dictionaries can yield unstable supports |
| Curved manifold exploration | Isomap or LLE | Neighbour graph quality and scalability |
| High-dimensional visualization | t-SNE or UMAP | Global spacing and cluster shapes are easy to over-interpret |
| Fast geometry-preserving compression | Random projection | Weak interpretability and stochastic distortion |
| Nonlinear reconstructive representation | Autoencoder | Capacity can memorize or preserve irrelevant detail |

A defensible workflow begins with the simplest representation aligned with the goal, compares it against unreduced features, fits every data-dependent step inside the training split, and reports what information was not preserved. A lower-dimensional picture is a model of the data, not the data themselves.

Official implementations and examples are available in the [scikit-learn decomposition guide](https://scikit-learn.org/stable/modules/decomposition.html), [manifold-learning guide](https://scikit-learn.org/stable/modules/manifold.html), [random-projection example](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_johnson_lindenstrauss_bound.html), and [UMAP parameter guide](https://umap-learn.readthedocs.io/en/latest/parameters.html).
